In [1]:
import os
import csv
import datetime as dt
import pandas as pd

# index를 날짜로 하는 데이터프레임을 price, divend에 대해 반환
def extract_stock_data(file_path, ticker):
    extract_datas = []

    with open(os.path.join(file_path, ticker + '.csv'), newline='') as csvfile:
        reader = csv.reader(csvfile)

        for row in reader:
            extract_datas.append(row)
        
    price_dates = list(map(dt.date.fromisoformat, extract_datas[0]))
    price_history = list(map(float, extract_datas[1]))
    divend_dates = list(map(dt.date.fromisoformat, extract_datas[2]))
    divend_history = list(map(float, extract_datas[3]))

    price_df = pd.DataFrame({'date': price_dates, 'price': price_history})
    divend_df = pd.DataFrame({'date': divend_dates, 'divend': divend_history})

    price_df.set_index('date', inplace=True)
    divend_df.set_index('date', inplace=True)

    return (price_df, divend_df)


In [2]:
def make_stock_data(file_path, tickers):
    prices = []
    divends = []

    for ticker in tickers:
        price_df, divend_df = extract_stock_data(file_path, ticker)
        price_df.rename(columns={'price': ticker}, inplace=True)
        divend_df.rename(columns={'divend': ticker}, inplace=True)
        prices.append(price_df)
        divends.append(divend_df)

    stock_price_df = pd.concat(prices, axis=1, sort=True)
    stock_divend_df = pd.concat(divends, axis=1, sort=True)

    return {'price': stock_price_df, 'divend': stock_divend_df}

In [ ]:
import itertools

class Portfolio:
    # stock_data
    def __init__(self, stock_data: dict, start_cash: float, target_ratio: dict, buy_ratio: float = 5.0, sell_ratio: float = 5.0):
        self.__stock_list = stock_data['price'].columns.to_list()
        self.__price_data = stock_data['price'].copy()
        self.__divend_data = stock_data['divend'].copy()
        self.__start_cash = start_cash
        # target_ratio의 구성 { stock명 : 목표 비중, ... }
        self.target_ratio = target_ratio
        self.buy_ratio = buy_ratio
        self.sell_ratio = sell_ratio
        self.__target_buy_ratio = {ticker: target_ratio[ticker] * buy_ratio / 100 for ticker in self.stock_list}
        self.__target_sell_ratio = {ticker: target_ratio[ticker] * sell_ratio / 100 for ticker in self.stock_list}
        # self.__target_buy_ratio = {ticker: min(target_ratio[ticker] * buy_ratio / 100, buy_ratio) for ticker in self.stock_list}
        # self.__target_sell_ratio = {ticker: min(target_ratio[ticker] * sell_ratio / 100, sell_ratio) for ticker in self.stock_list}
    

        self.__price_data.dropna(inplace=True)
        self.__avilable_date = self.price_data.iloc[0].name
        self.start_date = self.__avilable_date
        self.__divend_data = self.__divend_data[self.__divend_data.index >= self.start_date]
        
    @property
    def stock_list(self) -> list:
        return self.__stock_list
    @property
    def price_data(self) -> pd.DataFrame:
        return self.__price_data
    @property
    def divend_data(self) -> pd.DataFrame:
        return self.__divend_data
    # @property
    # def target_ratio(self) -> dict:
    #     return self.__target_ratio
    # @target_ratio.setter
    # def target_ratio(self, value):
    #     self.__target_ratio = value
    #     self.__target_buy_ratio = {ticker: min(value[ticker] * self.__buy_ratio / 100, self.__buy_ratio) for ticker in self.stock_list}
    #     self.__target_sell_ratio = {ticker: min(value[ticker] * self.__sell_ratio / 100, self.__sell_ratio) for ticker in self.stock_list}
    # @property
    # def buy_ratio(self) -> float:
    #     return self.__buy_ratio
    # @buy_ratio.setter
    # def buy_ratio(self, value):
    #     self.__buy_ratio = value
    #     self.__target_buy_ratio = {ticker: min(self.__target_ratio[ticker] * value / 100, value) for ticker in self.stock_list}
    # @property
    # def sell_ratio(self) -> float:  
    #     return self.__sell_ratio
    # @sell_ratio.setter
    # def sell_ratio(self, value):
    #     self.__sell_ratio = value
    #     self.__target_sell_ratio = {ticker: min(self.__target_ratio[ticker] * value / 100, value) for ticker in self.stock_list}

    def backtest(self):
        info_list =  ['number', 'value', 'weight']
        stock_info =  list(itertools.product(self.stock_list, info_list))

        col = [('total', 'value'), ('cash', 'value'), ('cash', 'weight')] + stock_info
        col = pd.MultiIndex.from_tuples(col)
        dates = self.price_data.index
        stat = pd.DataFrame(columns=col, index=dates)

        # 첫 값 설정
        first_date = dates[0]
        total_value = self.__start_cash
        for ticker in self.stock_list:
            stat.loc[first_date, ('total', 'value')] = total_value
            stat.loc[first_date, ('cash', 'value')] = 0
            stat.loc[first_date, ('cash', 'weight')] = 0
            stat.loc[first_date, [ticker]] = self.__ideal_nvw(total_value, ticker, first_date)
            
        for i in range(1, len(dates)):
            cash_value = stat.loc[dates[i - 1]][('cash', 'value')]
            total_value = cash_value
            stat.loc[dates[i], ('cash', 'value')] = stat.loc[dates[i - 1]][('cash', 'value')]
            # 가격 변동 처리
            prev_num = stat.loc[dates[i - 1]][:, 'number']
            prev_num_pv = (prev_num * self.price_data.loc[dates[i]]).round(3)
            total_value += prev_num_pv.sum()
            # 배당금 처리
            divend_df = self.acculate_divend(dates[i-1], dates[i])
            divend = (divend_df.sum() * prev_num).sum().round(3)
            cash_value += divend
            total_value += divend
            stat.loc[dates[i], ('cash', 'value')] = cash_value
            stat.loc[dates[i], ('total', 'value')] = total_value

            # 변동에 따른 리밸런싱 계산
            prev_num_ratio = (prev_num_pv / total_value * 100).round(2)
            target_ratio_sr = pd.Series(self.target_ratio)
            ratio_diff = (prev_num_ratio - target_ratio_sr)

            need_sell = ratio_diff[ratio_diff >= pd.Series(self.__target_sell_ratio)].index.to_list()
            need_buy = ratio_diff[ratio_diff <= -pd.Series(self.__target_buy_ratio)].index.to_list()
            need_trade = need_sell + need_buy
            for ticker in self.stock_list:
                if ticker in need_trade:
                    stat.loc[dates[i], ticker] = self.__ideal_nvw(total_value, ticker, dates[i])
                    stat.loc[dates[i], ('cash', 'value')] += (ratio_diff[ticker] * total_value * 0.01).round(3)
                else:
                    stat.loc[dates[i], (ticker, 'number')] = stat.loc[dates[i - 1], (ticker, 'number')]
                    stat.loc[dates[i], (ticker, 'value')] = prev_num_pv[ticker]
                    stat.loc[dates[i], (ticker, 'weight')] = prev_num_ratio[ticker]
            
            stat.loc[dates[i], ('cash', 'weight')] = (stat.loc[dates[i]][('cash', 'value')] / total_value * 100).round(2)

        return stat

    # 범위는 (start_date, end_date]
    def acculate_divend(self, start_date, end_date):
        divend_df = self.divend_data[(self.divend_data.index > start_date) & (self.divend_data.index <= end_date)]
        return divend_df
    
    def __ideal_nvw(self, total_value, ticker, date):
        target_ratio = self.target_ratio[ticker] * 0.01
        value = round(total_value * target_ratio, 3)
        price = self.price_data.loc[date][ticker]
        number = round(value / price, 6)
        return (number, value, target_ratio * 100)
    
    # def __real_nvw(self, total_value, ticker, date):
    #     price = self.price_data.loc[date][ticker]
    #     ideal = self.__ideal_nvw(total_value, ticker, date)
        


In [333]:
# stock 불러와서 데이터프레임화 하기
file_path = './etf'
tickers = ['QQQ', 'DGRW', 'SCHD']

stock_data = make_stock_data(file_path, tickers)

In [337]:
portfolio = Portfolio(stock_data, 10000, {'QQQ': 50, 'DGRW': 30, 'SCHD': 20}, buy_ratio=5, sell_ratio=5)

backtest_stat = portfolio.backtest()
pd.set_option('display.max_rows', None)
print(backtest_stat)

                total      cash               QQQ                    \
                value     value weight     number      value weight   
date                                                                  
2013-05-20      10000         0      0  68.110612     5000.0   50.0   
2013-05-27   9927.019       0.0    0.0  68.110612   4989.102  50.26   
2013-06-03   9980.815       0.0    0.0  68.110612   4998.638  50.08   
2013-06-10   9860.797       0.0    0.0  68.110612   4923.035  49.93   
2013-06-17    9640.15       0.0    0.0  68.110612    4797.03  49.76   
2013-06-24   9743.176     33.16   0.34  68.110612   4854.243  49.82   
2013-07-01   9907.446     33.16   0.33  68.110612   4943.468   49.9   
2013-07-08  10231.609     33.16   0.32  68.110612   5128.729  50.13   
2013-07-15  10192.768     33.16   0.33  68.110612   5080.371  49.84   
2013-07-22  10243.234    37.753   0.37  68.110612   5133.497  50.12   
2013-07-29  10414.113    37.753   0.36  68.110612   5242.474  50.34   
2013-0